#Inicializaciones

In [ ]:
import pandas as pd
from google.colab import drive
import email
from email.utils import parsedate_to_datetime
import re
import json
import time

In [ ]:
#Conectar con Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Conectar con Gemini
from google import genai
from google.colab import userdata

api_key = userdata.get('GOOGLE_API_KEY')
client_gemini = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.1-flash-lite"

In [ ]:
# Instalamos ChromaDB
!pip install chromadb -q

import chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [ ]:
import chromadb

cliente_completos_v2 = chromadb.PersistentClient(path="/content/drive/MyDrive/TFM/chroma_beck_A_v2")
collection_completos_v2 = cliente_completos_v2.get_or_create_collection(
    name="beck_mails_completos_v2",
    metadata={"hnsw:space": "cosine"}
)

print(collection_completos_v2.count())

3460


In [ ]:
#Instalar Ecologist
!pip install "ecologits[google-genai]" -q

from ecologits import EcoLogits
EcoLogits.init(providers=["google_genai"])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 663.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.1/88.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 1.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 1.75.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [ ]:
df_beck_final = pd.read_csv("/content/drive/MyDrive/TFM/dataset_beck_final.csv")

In [ ]:
df_beck_final.head(3)

,subject,from_addr,fecha_primer_mensaje,fecha_ultimo_mensaje,body,n_mensajes
0,Key Contacts in Your Area - December 20 - 31,sally.beck@enron.com,1999-12-13 22:25:00+00:00,1999-12-13 22:25:00+00:00,[1999-12-13 22:25:00+00:00] sally.beck@enron.c...,1
1,Re: Key Contacts in Your Area - December 20 - 31,sally.beck@enron.com,1999-12-15 14:14:00+00:00,1999-12-15 14:14:00+00:00,[1999-12-15 14:14:00+00:00] sally.beck@enron.c...,1
2,Re: Rotations,sally.beck@enron.com,1999-12-13 20:45:00+00:00,1999-12-16 08:42:00+00:00,[1999-12-13 20:45:00+00:00] sally.beck@enron.c...,2


In [ ]:
df_preguntas = pd.read_csv("/content/drive/MyDrive/TFM/banco_preguntas_v2.csv")

In [ ]:
!pip install transformers -q

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
n_tokens = df_beck_final["body"].apply(lambda x: len(tokenizer.encode(x, truncation=False)))

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (529 > 512). Running this sequence through the model will result in indexing errors


#Indexación A_2

In [ ]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(modelo.max_seq_length)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

256


In [ ]:
!pip install langchain_experimental langchain_community sentence-transformers -q

from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

chunker = SemanticChunker(embeddings, min_chunk_size=85)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 1.75.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


/tmp/ipykernel_1455/4111997102.py:3: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
/tmp/ipykernel_1455/4111997102.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
correos_largos = df_beck_final[n_tokens > 256]
muestra = correos_largos.sample(5, random_state=42)

for idx, fila in muestra.iterrows():
    print(f"=== Correo {idx} (original: {n_tokens.loc[idx]} tokens) ===")
    chunks = chunker.split_text(fila["body"])
    print(f"Número de chunks generados: {len(chunks)}")
    for i, c in enumerate(chunks):
        tokens_chunk = len(tokenizer.encode(c, add_special_tokens=False))
        marca = "⚠️ SUPERA 256" if tokens_chunk > 256 else ""
        print(f"  Chunk {i}: {tokens_chunk} tokens {marca}")
    print()

=== Correo 1521 (original: 593 tokens) ===
Número de chunks generados: 2
  Chunk 0: 377 tokens ⚠️ SUPERA 256
  Chunk 1: 214 tokens 

=== Correo 424 (original: 279 tokens) ===
Número de chunks generados: 2
  Chunk 0: 177 tokens 
  Chunk 1: 100 tokens 

=== Correo 1836 (original: 275 tokens) ===
Número de chunks generados: 2
  Chunk 0: 185 tokens 
  Chunk 1: 88 tokens 

=== Correo 1925 (original: 604 tokens) ===
Número de chunks generados: 3
  Chunk 0: 354 tokens ⚠️ SUPERA 256
  Chunk 1: 185 tokens 
  Chunk 2: 63 tokens 

=== Correo 500 (original: 309 tokens) ===
Número de chunks generados: 2
  Chunk 0: 254 tokens 
  Chunk 1: 53 tokens 



In [ ]:
idx_extremo = n_tokens.idxmax()
texto_extremo = df_beck_final.loc[idx_extremo, "body"]
print(f"Índice: {idx_extremo}, tokens: {n_tokens.loc[idx_extremo]}")

try:
    chunks_extremo = chunker.split_text(texto_extremo)
    print(f"✅ Funcionó. Número de chunks generados: {len(chunks_extremo)}")
    tokens_por_chunk = [len(tokenizer.encode(c, add_special_tokens=False)) for c in chunks_extremo]
    print(f"Tokens por chunk: {tokens_por_chunk}")
    print(f"Chunks que superan 256: {sum(1 for t in tokens_por_chunk if t > 256)}")
except Exception as e:
    print(f"❌ Falló: {type(e).__name__}: {e}")

Índice: 1915, tokens: 25058
✅ Funcionó. Número de chunks generados: 32
Tokens por chunk: [1186, 1806, 1519, 127, 46, 882, 635, 321, 782, 139, 76, 114, 112, 408, 124, 547, 1259, 1253, 1239, 1488, 1629, 375, 836, 857, 314, 2250, 905, 793, 827, 1625, 462, 120]
Chunks que superan 256: 24


In [ ]:
def fusionar_ultimo_si_pequeno(chunks, min_chars=85):
    """
    Corrige un caso no cubierto por SemanticChunker: el último fragmento que
    devuelve split_text() no pasa por su comprobación interna de min_chunk_size,
    así que puede quedar un resto degenerado (ej. una firma suelta) tras el
    último punto de corte semántico. Si eso ocurre, se fusiona con el
    fragmento anterior en vez de indexarse como documento independiente.
    """
    if len(chunks) > 1 and len(chunks[-1]) < min_chars:
        chunks[-2] = chunks[-2] + " " + chunks[-1]
        chunks = chunks[:-1]
    return chunks


MAX_TOKENS_CHUNK = 250
OVERLAP_CHUNK = 25

def dividir_por_tokens(chunk, max_tokens=MAX_TOKENS_CHUNK, overlap=OVERLAP_CHUNK):
    """
    Red de seguridad mecánica: SemanticChunker no impone ningún límite máximo
    de tamaño, solo agrupa por similitud. Si un fragmento semántico aún supera
    el límite real de tokens del modelo de embeddings (256), se corta por
    ventana de tamaño fijo, con solapamiento entre trozos consecutivos para
    no perder contexto justo en el punto de corte. Si el fragmento ya cabe
    dentro del límite, se devuelve sin modificar.
    """
    tokens = tokenizer.encode(chunk, add_special_tokens=False)
    if len(tokens) <= max_tokens:
        return [chunk]

    sub_chunks = []
    paso = max_tokens - overlap
    for i in range(0, len(tokens), paso):
        sub_tokens = tokens[i:i + max_tokens]
        sub_chunks.append(tokenizer.decode(sub_tokens))
        if i + max_tokens >= len(tokens):
            break
    return sub_chunks

In [ ]:
documents = []
ids = []
metadatas = []

for idx, fila in df_beck_final.iterrows():
    n_tok = n_tokens.loc[idx]

    if n_tok <= 256:
        documents.append(fila["body"])
        ids.append(str(idx))
        metadatas.append({
            "subject": fila["subject"],
            "from_addr": fila["from_addr"],
            "fecha_primer_mensaje": str(fila["fecha_primer_mensaje"]),
            "fecha_ultimo_mensaje": str(fila["fecha_ultimo_mensaje"]),
            "n_mensajes": int(fila["n_mensajes"]),
            "es_chunk": False,
        })
    else:
        chunks_semanticos = chunker.split_text(fila["body"])
        chunks_semanticos = fusionar_ultimo_si_pequeno(chunks_semanticos)

        chunks_finales = []
        for c in chunks_semanticos:
            tokens_c = len(tokenizer.encode(c, add_special_tokens=False))
            if tokens_c > 256:
                chunks_finales.extend(dividir_por_tokens(c))
            else:
                chunks_finales.append(c)

        for i, chunk in enumerate(chunks_finales):
            documents.append(chunk)
            ids.append(f"{idx}_chunk{i}")
            metadatas.append({
                "subject": fila["subject"],
                "from_addr": fila["from_addr"],
                "fecha_primer_mensaje": str(fila["fecha_primer_mensaje"]),
                "fecha_ultimo_mensaje": str(fila["fecha_ultimo_mensaje"]),
                "n_mensajes": int(fila["n_mensajes"]),
                "es_chunk": True,
                "documento_original_id": str(idx),
                "chunk_index": i,
                "total_chunks": len(chunks_finales),
            })

    if idx % 200 == 0:
        print(f"Progreso: {idx}/{len(df_beck_final)}")

print(f"\nTotal de documentos a indexar: {len(documents)}")

Progreso: 0/2229
Progreso: 200/2229
Progreso: 400/2229
Progreso: 600/2229
Progreso: 800/2229
Progreso: 1000/2229
Progreso: 1200/2229
Progreso: 1400/2229
Progreso: 1600/2229
Progreso: 1800/2229
Progreso: 2000/2229
Progreso: 2200/2229

Total de documentos a indexar: 3460


In [ ]:
import chromadb

cliente_completos_v2 = chromadb.PersistentClient(path="/content/drive/MyDrive/TFM/chroma_beck_A_v2")
collection_completos_v2 = cliente_completos_v2.get_or_create_collection(
    name="beck_mails_completos_v2",
    metadata={"hnsw:space": "cosine"}
)

print(collection_completos_v2.count())

3460


In [ ]:
import time

inicio_indexacion_a_v2 = time.time()

collection_completos_v2.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)

tiempo_indexacion_a_v2 = time.time() - inicio_indexacion_a_v2

print(collection_completos_v2.count())
print(f"Tiempo de indexación: {tiempo_indexacion_a_v2:.2f} segundos")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 105MiB/s]


3460
Tiempo de indexación: 540.90 segundos


##Comprobaciones

In [ ]:
consultas_prueba = {
    "manager_west": "Who was appointed as manager of the West region?",
    "bob_impacto": "Who did Sally ask to assess the impact of losing Bob?",
    "astros_tickets": "How many Astros tickets did Sally receive for the exhibition game on April 1?",
    "generica_oficina": "Sally is out of the office",
    "control_mundial": "Who won the 2026 World Cup?",
    "control_oro": "What is the chemical symbol for gold?",
    "control_inception": "Who directed the movie Inception?",
}

for tipo, consulta in consultas_prueba.items():
    resultado = collection_completos_v2.query(query_texts=[consulta], n_results=5)
    print(f"--- {tipo}: \"{consulta}\" ---")
    for doc, dist in zip(resultado["documents"][0], resultado["distances"][0]):
        print(f"  Distancia: {dist:.4f} | {doc[:100]}")
    print()

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 80.8MiB/s]


--- manager_west: "Who was appointed as manager of the West region?" ---
  Distancia: 0.4117 | [2000-04-18 14:17:00+00:00] sally.beck@enron.com:
We did talk about the creation of these new positi
  Distancia: 0.5719 | The producer focus in the Western U.S. and Texas will remain unchanged reporting  to Mark Whitt and 
  Distancia: 0.6228 | He will work actively with the operation=
s=20
staff in Enron Europe to achieve this goal. =20

(FER
  Distancia: 0.6323 | [2001-10-25 02:22:28+00:00] no.address@enron.com:
Our natural gas business continues to benefit from
  Distancia: 0.6534 | Strengthen Mid and Back Office Talent and Controls for Global Risk 
Management Operations

A. Establ

--- bob_impacto: "Who did Sally ask to assess the impact of losing Bob?" ---
  Distancia: 0.5035 | [2000-06-13 18:40:00+00:00] sally.beck@enron.com:
Bob is a good, solid player. He won't change the w
  Distancia: 0.5393 | [2000-06-12 17:41:00+00:00] sally.beck@enron.com:
I am not close enough to know Bob's exa

#Experimentos A_2

In [ ]:
def agente_generador_respuesta_k1(pregunta, resultado_chroma, umbral_distancia=0.6):
    documento = resultado_chroma["documents"][0][0]
    distancia = resultado_chroma["distances"][0][0]

    if distancia > umbral_distancia:
        return {
            "respuesta": "No information available in the provided emails.",
            "prompt_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "tiempo_ejecucion": 0.0,
            "energia_wh": 0.0,
            "gwp_mgco2eq": 0.0,
        }

    prompt = f"""Eres el asistente personal de correo electrónico de Sally Beck. Sally es quien te está haciendo la pregunta directamente.

Respóndele a Sally en segunda persona ("you", "your"), como lo haría un asistente en una conversación normal.

Responde la pregunta ÚNICAMENTE usando la información del siguiente correo.

CORREO:
{documento}

PREGUNTA: {pregunta}

Responde de forma directa, en el idioma del correo.
"""

    inicio = time.time()

    respuesta = client_gemini.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={"temperature": 0.2}
    )

    tiempo_ejecucion = time.time() - inicio
    energia = respuesta.impacts.energy.value
    gwp = respuesta.impacts.gwp.value

    return {
        "respuesta": respuesta.text.strip(),
        "prompt_tokens": respuesta.usage_metadata.prompt_token_count,
        "output_tokens": respuesta.usage_metadata.candidates_token_count,
        "total_tokens": respuesta.usage_metadata.total_token_count,
        "tiempo_ejecucion": round(tiempo_ejecucion, 3),
        "energia_wh": round(energia.mean * 1000, 4),
        "gwp_mgco2eq": round(gwp.mean * 1_000_000, 4),
    }

In [ ]:
resultados_a_k1_v2 = []

for i, fila in df_preguntas.iterrows():
    pregunta = fila["pregunta"]

    resultado_chroma = collection_completos_v2.query(query_texts=[pregunta], n_results=1)
    respuesta = agente_generador_respuesta_k1(pregunta, resultado_chroma)

    resultados_a_k1_v2.append({
        "id_pregunta": i,
        "id_correo": fila["id_correo"],
        "tamano_correo": fila["tamano_correo"],
        "pregunta": pregunta,
        "respuesta_esperada": fila["respuesta_esperada"],
        "respuesta_generada": respuesta["respuesta"],
        "prompt_tokens": respuesta["prompt_tokens"],
        "output_tokens": respuesta["output_tokens"],
        "total_tokens": respuesta["total_tokens"],
        "tiempo_ejecucion": respuesta["tiempo_ejecucion"],
        "energia_wh": respuesta["energia_wh"],
        "gwp_mgco2eq": respuesta["gwp_mgco2eq"],
    })

    if i % 20 == 0:
        pd.DataFrame(resultados_a_k1_v2).to_csv("/content/drive/MyDrive/TFM/experimento_A_k1_v2.csv", index=False)
        print(f"Progreso: {i}/{len(df_preguntas)}")

    time.sleep(4)

pd.DataFrame(resultados_a_k1_v2).to_csv("/content/drive/MyDrive/TFM/experimento_A_k1_v2.csv", index=False)
print("Terminado")

Progreso: 0/200
Progreso: 20/200
Progreso: 40/200
Progreso: 60/200
Progreso: 80/200
Progreso: 100/200
Progreso: 120/200
Progreso: 140/200
Progreso: 160/200


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
print(len(resultados_a_k1_v2))

178


In [ ]:
pd.DataFrame(resultados_a_k1_v2).to_csv("/content/drive/MyDrive/TFM/experimento_A_k1_v2.csv", index=False)
print(len(resultados_a_k1_v2))

178


In [ ]:
for i, fila in df_preguntas.iloc[178:].iterrows():
    pregunta = fila["pregunta"]

    resultado_chroma = collection_completos_v2.query(query_texts=[pregunta], n_results=1)
    respuesta = agente_generador_respuesta_k1(pregunta, resultado_chroma)

    resultados_a_k1_v2.append({
        "id_pregunta": i,
        "id_correo": fila["id_correo"],
        "tamano_correo": fila["tamano_correo"],
        "pregunta": pregunta,
        "respuesta_esperada": fila["respuesta_esperada"],
        "respuesta_generada": respuesta["respuesta"],
        "prompt_tokens": respuesta["prompt_tokens"],
        "output_tokens": respuesta["output_tokens"],
        "total_tokens": respuesta["total_tokens"],
        "tiempo_ejecucion": respuesta["tiempo_ejecucion"],
        "energia_wh": respuesta["energia_wh"],
        "gwp_mgco2eq": respuesta["gwp_mgco2eq"],
    })

    if len(resultados_a_k1_v2) % 20 == 0:
        pd.DataFrame(resultados_a_k1_v2).to_csv("/content/drive/MyDrive/TFM/experimento_A_k1_v2.csv", index=False)
        print(f"Progreso: {len(resultados_a_k1_v2)}/200")

    time.sleep(4)

pd.DataFrame(resultados_a_k1_v2).to_csv("/content/drive/MyDrive/TFM/experimento_A_k1_v2.csv", index=False)
print("Terminado")

Progreso: 180/200
Progreso: 200/200
Terminado


In [ ]:
verificacion_a_k1 = pd.read_csv("/content/drive/MyDrive/TFM/experimento_A_k1_v2.csv")
print(verificacion_a_k1.shape)
print(verificacion_a_k1.isna().sum())
print()
display(verificacion_a_k1.sample(5))

(200, 12)
id_pregunta            0
id_correo             20
tamano_correo          0
pregunta               0
respuesta_esperada     0
respuesta_generada     0
prompt_tokens          0
output_tokens          0
total_tokens           0
tiempo_ejecucion       0
energia_wh             0
gwp_mgco2eq            0
dtype: int64



,id_pregunta,id_correo,tamano_correo,pregunta,respuesta_esperada,respuesta_generada,prompt_tokens,output_tokens,total_tokens,tiempo_ejecucion,energia_wh,gwp_mgco2eq
126,126,383.0,largo,"What is the phone number for Beverly Scott, wh...",Beverly Scott's work phone number is 281-444-0...,The email does not provide Beverly Scott's pho...,352,28,380,2.319,0.0101,4.5675
51,51,823.0,intermedio,For which employees did I complete internal fe...,"I completed feedback forms for Brent Price, Bo...","As of December 29, 2000, you completed interna...",337,34,371,1.575,0.0113,4.9821
29,29,986.0,corto,Why will I be unable to attend a meeting on Mo...,I will be unable to attend because I will be a...,The email does not state the reason why you wi...,160,27,187,0.893,0.0083,3.5666
44,44,29.0,corto,For which employee did I submit a completed pr...,I submitted the profile for Brent Price.,You submitted the completed profile for Brent ...,174,9,183,0.625,0.0034,1.5445
19,19,655.0,corto,What did I request regarding my hotel room res...,I requested that my hotel reservation be for a...,You didn't make a specific request regarding a...,364,36,400,0.841,0.0105,4.3749


In [ ]:
def agente_generador_respuesta_multi(pregunta, resultado_chroma, umbral_distancia=0.6):
    documentos = resultado_chroma["documents"][0]
    distancias = resultado_chroma["distances"][0]

    documentos_relevantes = []
    for doc, dist in zip(documentos, distancias):
        if dist <= umbral_distancia:
            documentos_relevantes.append(doc)

    if not documentos_relevantes:
        return {
            "respuesta": "No information available in the provided emails.",
            "prompt_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "tiempo_ejecucion": 0.0,
            "energia_wh": 0.0,
            "gwp_mgco2eq": 0.0,
        }

    contexto = "\n\n---\n\n".join(documentos_relevantes)
    n_correos = len(documentos_relevantes)

    prompt = f"""Eres el asistente personal de correo electrónico de Sally Beck. Sally es quien te está haciendo la pregunta directamente.

Respóndele a Sally en segunda persona ("you", "your"), como lo haría un asistente en una conversación normal.

Recibirás {n_correos} correos recuperados de una base de datos vectorial por su similitud semántica con la pregunta. Estos correos no forman necesariamente un mismo hilo ni están relacionados entre sí: trátalos como fuentes independientes.

Responde la pregunta ÚNICAMENTE usando la información de los correos proporcionados.

CORREOS:
{contexto}

PREGUNTA: {pregunta}

Responde de forma directa, en el idioma de los correos.
"""

    inicio = time.time()

    respuesta = client_gemini.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={"temperature": 0.2}
    )

    tiempo_ejecucion = time.time() - inicio
    energia = respuesta.impacts.energy.value
    gwp = respuesta.impacts.gwp.value

    return {
        "respuesta": respuesta.text.strip(),
        "prompt_tokens": respuesta.usage_metadata.prompt_token_count,
        "output_tokens": respuesta.usage_metadata.candidates_token_count,
        "total_tokens": respuesta.usage_metadata.total_token_count,
        "tiempo_ejecucion": round(tiempo_ejecucion, 3),
        "energia_wh": round(energia.mean * 1000, 4),
        "gwp_mgco2eq": round(gwp.mean * 1_000_000, 4),
    }

In [ ]:
resultados_a_k3_v2 = []

inicio = len(resultados_a_k3_v2)
fin = inicio + 5

for i, fila in df_preguntas.iloc[inicio:fin].iterrows():
    pregunta = fila["pregunta"]

    resultado_chroma = collection_completos_v2.query(query_texts=[pregunta], n_results=3)
    respuesta = agente_generador_respuesta_multi(pregunta, resultado_chroma)

    resultados_a_k3_v2.append({
        "id_pregunta": i,
        "id_correo": fila["id_correo"],
        "tamano_correo": fila["tamano_correo"],
        "pregunta": pregunta,
        "respuesta_esperada": fila["respuesta_esperada"],
        "respuesta_generada": respuesta["respuesta"],
        "prompt_tokens": respuesta["prompt_tokens"],
        "output_tokens": respuesta["output_tokens"],
        "total_tokens": respuesta["total_tokens"],
        "tiempo_ejecucion": respuesta["tiempo_ejecucion"],
        "energia_wh": respuesta["energia_wh"],
        "gwp_mgco2eq": respuesta["gwp_mgco2eq"],
    })

    time.sleep(6)

pd.DataFrame(resultados_a_k3_v2).to_csv("/content/drive/MyDrive/TFM/experimento_A_k3_v2.csv", index=False)
print(f"Progreso: {len(resultados_a_k3_v2)}/200")

NameError: name 'resultados_a_k3_v2' is not defined

In [ ]:
print(len(resultados_a_k3_v2))

NameError: name 'resultados_a_k3_v2' is not defined

In [ ]:
pd.DataFrame(resultados_a_k3_v2).to_csv("/content/drive/MyDrive/TFM/experimento_A_k3_v2.csv", index=False)
print(len(resultados_a_k3_v2))

200


In [ ]:
df_verificacion = pd.read_csv("/content/drive/MyDrive/TFM/experimento_A_k3_v2.csv")
print("Forma:", df_verificacion.shape)
print()
print("Nulos por columna:")
print(df_verificacion.isna().sum())
print()
print("¿Índices duplicados?", df_verificacion["id_pregunta"].duplicated().any())
print("¿Están en orden 0,1,2...?", (df_verificacion["id_pregunta"] == range(len(df_verificacion))).all())
print()
print("Longitud mínima de respuesta_generada:", df_verificacion["respuesta_generada"].str.len().min())
print("¿Alguna respuesta vacía o muy corta (<5 caracteres)?", (df_verificacion["respuesta_generada"].str.len() < 5).sum())

Forma: (200, 12)

Nulos por columna:
id_pregunta            0
id_correo             20
tamano_correo          0
pregunta               0
respuesta_esperada     0
respuesta_generada     0
prompt_tokens          0
output_tokens          0
total_tokens           0
tiempo_ejecucion       0
energia_wh             0
gwp_mgco2eq            0
dtype: int64

¿Índices duplicados? False
¿Están en orden 0,1,2...? True

Longitud mínima de respuesta_generada: 35
¿Alguna respuesta vacía o muy corta (<5 caracteres)? 0


In [ ]:
resultados_a_k5_v2 = []

for i, fila in df_preguntas.iterrows():
    pregunta = fila["pregunta"]

    resultado_chroma = collection_completos_v2.query(query_texts=[pregunta], n_results=5)
    respuesta = agente_generador_respuesta_multi(pregunta, resultado_chroma)

    resultados_a_k5_v2.append({
        "id_pregunta": i,
        "id_correo": fila["id_correo"],
        "tamano_correo": fila["tamano_correo"],
        "pregunta": pregunta,
        "respuesta_esperada": fila["respuesta_esperada"],
        "respuesta_generada": respuesta["respuesta"],
        "prompt_tokens": respuesta["prompt_tokens"],
        "output_tokens": respuesta["output_tokens"],
        "total_tokens": respuesta["total_tokens"],
        "tiempo_ejecucion": respuesta["tiempo_ejecucion"],
        "energia_wh": respuesta["energia_wh"],
        "gwp_mgco2eq": respuesta["gwp_mgco2eq"],
    })

    if i % 20 == 0:
        pd.DataFrame(resultados_a_k5_v2).to_csv("/content/drive/MyDrive/TFM/experimento_A_k5_v2.csv", index=False)
        print(f"Progreso: {i}/{len(df_preguntas)}")

    time.sleep(4)

pd.DataFrame(resultados_a_k5_v2).to_csv("/content/drive/MyDrive/TFM/experimento_A_k5_v2.csv", index=False)
print("Terminado")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 83.3MiB/s]


Progreso: 0/200
Progreso: 20/200


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
df_recuperado = pd.read_csv("/content/drive/MyDrive/TFM/experimento_A_k5_v2.csv")
resultados_a_k5_v2 = df_recuperado.to_dict("records")
print(len(resultados_a_k5_v2))

62


In [ ]:
inicio = len(resultados_a_k5_v2)
fin = inicio + 5

for i, fila in df_preguntas.iloc[inicio:fin].iterrows():
    pregunta = fila["pregunta"]

    resultado_chroma = collection_completos_v2.query(query_texts=[pregunta], n_results=5)
    respuesta = agente_generador_respuesta_multi(pregunta, resultado_chroma)

    resultados_a_k5_v2.append({
        "id_pregunta": i,
        "id_correo": fila["id_correo"],
        "tamano_correo": fila["tamano_correo"],
        "pregunta": pregunta,
        "respuesta_esperada": fila["respuesta_esperada"],
        "respuesta_generada": respuesta["respuesta"],
        "prompt_tokens": respuesta["prompt_tokens"],
        "output_tokens": respuesta["output_tokens"],
        "total_tokens": respuesta["total_tokens"],
        "tiempo_ejecucion": respuesta["tiempo_ejecucion"],
        "energia_wh": respuesta["energia_wh"],
        "gwp_mgco2eq": respuesta["gwp_mgco2eq"],
    })

    time.sleep(10)

pd.DataFrame(resultados_a_k5_v2).to_csv("/content/drive/MyDrive/TFM/experimento_A_k5_v2.csv", index=False)
print(f"Progreso: {len(resultados_a_k5_v2)}/200")

Progreso: 200/200


In [ ]:
print(len(resultados_a_k5_v2))

200


In [ ]:
pd.DataFrame(resultados_a_k5_v2).to_csv("/content/drive/MyDrive/TFM/experimento_A_k5_v2.csv", index=False)
print(len(resultados_a_k5_v2))

200


In [ ]:
df_verificacion = pd.read_csv("/content/drive/MyDrive/TFM/experimento_A_k5_v2.csv")
print("Forma:", df_verificacion.shape)
print()
print("Nulos por columna:")
print(df_verificacion.isna().sum())
print()
print("¿Índices duplicados?", df_verificacion["id_pregunta"].duplicated().any())
print("¿Están en orden 0,1,2...?", (df_verificacion["id_pregunta"] == range(len(df_verificacion))).all())
print()
print("Longitud mínima de respuesta_generada:", df_verificacion["respuesta_generada"].str.len().min())
print("¿Alguna respuesta vacía o muy corta (<5 caracteres)?", (df_verificacion["respuesta_generada"].str.len() < 5).sum())

Forma: (200, 12)

Nulos por columna:
id_pregunta            0
id_correo             20
tamano_correo          0
pregunta               0
respuesta_esperada     0
respuesta_generada     0
prompt_tokens          0
output_tokens          0
total_tokens           0
tiempo_ejecucion       0
energia_wh             0
gwp_mgco2eq            0
dtype: int64

¿Índices duplicados? False
¿Están en orden 0,1,2...? True

Longitud mínima de respuesta_generada: 41
¿Alguna respuesta vacía o muy corta (<5 caracteres)? 0


##Análisis de Resultados

K=1

In [ ]:
df_claude_k1_v2 = pd.read_csv("experimento_A_k1_v2_evaluado_claude.csv")
df_gemini_k1_v2 = pd.read_csv("experimento_A_k1_v2_evaluado_Gemini.csv")

print("Tasa de acierto Claude:", df_claude_k1_v2["acierto"].mean())
print("Tasa de acierto Gemini:", df_gemini_k1_v2["acierto"].mean())

coincidencias = (df_claude_k1_v2["acierto"] == df_gemini_k1_v2["acierto"]).sum()
print(f"Coinciden en {coincidencias}/200 filas")

discrepancias = df_claude_k1_v2[df_claude_k1_v2["acierto"] != df_gemini_k1_v2["acierto"]]
print(f"Discrepancias: {len(discrepancias)}")
print(discrepancias["id_pregunta"].tolist())

Tasa de acierto Claude: 0.56
Tasa de acierto Gemini: 0.535
Coinciden en 189/200 filas
Discrepancias: 11
[14, 31, 63, 71, 79, 105, 113, 128, 134, 137, 145]


In [ ]:
df_resultados_k1_v2 = df_claude_k1_v2.drop(columns=["acierto"]).copy()
df_resultados_k1_v2["acierto_claude"] = df_claude_k1_v2["acierto"]
df_resultados_k1_v2["acierto_gemini"] = df_gemini_k1_v2["acierto"]

df_resultados_k1_v2.to_csv("/content/drive/MyDrive/TFM/experimento_A_k1_v2_evaluado.csv", index=False)
print(df_resultados_k1_v2.shape)

(200, 14)


In [ ]:
df_resultados_a_k1_v2 = pd.read_csv("/content/drive/MyDrive/TFM/experimento_A_k1_v2_evaluado.csv")

In [ ]:
resumen_por_categoria = df_resultados_a_k1_v2.groupby("tamano_correo")["acierto_claude"].agg(["sum", "count", "mean"])
resumen_por_categoria.columns = ["aciertos", "total", "tasa_acierto"]
print("Claude:")
print(resumen_por_categoria)
print()

resumen_por_categoria = df_resultados_a_k1_v2.groupby("tamano_correo")["acierto_gemini"].agg(["sum", "count", "mean"])
resumen_por_categoria.columns = ["aciertos", "total", "tasa_acierto"]
print("Gemini:")
print(resumen_por_categoria)

Claude:
               aciertos  total  tasa_acierto
tamano_correo                               
corto                28     50      0.560000
intermedio           34     50      0.680000
largo                17     50      0.340000
multi_mensaje        13     30      0.433333
sin_respuesta        20     20      1.000000

Gemini:
               aciertos  total  tasa_acierto
tamano_correo                               
corto                26     50      0.520000
intermedio           31     50      0.620000
largo                17     50      0.340000
multi_mensaje        13     30      0.433333
sin_respuesta        20     20      1.000000


In [ ]:
largas_k1 = df_resultados_a_k1_v2.loc[100:149].copy()
largas_k1["necesito_chunking"] = largas_k1["id_correo"].apply(lambda x: n_tokens.loc[int(float(x))] > 256)

resumen_chunking_k1_claude = largas_k1.groupby("necesito_chunking")["acierto_claude"].agg(["sum", "count", "mean"])
resumen_chunking_k1_claude.columns = ["aciertos", "total", "tasa_acierto"]
print("Claude:")
print(resumen_chunking_k1_claude)
print()

resumen_chunking_k1_gemini = largas_k1.groupby("necesito_chunking")["acierto_gemini"].agg(["sum", "count", "mean"])
resumen_chunking_k1_gemini.columns = ["aciertos", "total", "tasa_acierto"]
print("Gemini:")
print(resumen_chunking_k1_gemini)

Claude:
                   aciertos  total  tasa_acierto
necesito_chunking                               
False                     9     15      0.600000
True                      8     35      0.228571

Gemini:
                   aciertos  total  tasa_acierto
necesito_chunking                               
False                     8     15      0.533333
True                      9     35      0.257143


K=3

In [ ]:
df_claude_k3_v2 = pd.read_csv("experimento_A_k3_v2_evaluado_claude.csv")
df_gemini_k3_v2 = pd.read_csv("experimento_A_k3_v2_evaluado_Gemini.csv")

print("Tasa de acierto Claude:", df_claude_k3_v2["acierto"].mean())
print("Tasa de acierto Gemini:", df_gemini_k3_v2["acierto"].mean())

coincidencias = (df_claude_k3_v2["acierto"] == df_gemini_k3_v2["acierto"]).sum()
print(f"Coinciden en {coincidencias}/200 filas")

discrepancias = df_claude_k3_v2[df_claude_k3_v2["acierto"] != df_gemini_k3_v2["acierto"]]
print(f"Discrepancias: {len(discrepancias)}")
print(discrepancias["id_pregunta"].tolist())

Tasa de acierto Claude: 0.695
Tasa de acierto Gemini: 0.68
Coinciden en 191/200 filas
Discrepancias: 9
[12, 27, 30, 45, 63, 79, 105, 134, 142]


In [ ]:
df_resultados_k3_v2 = df_claude_k3_v2.drop(columns=["acierto"]).copy()
df_resultados_k3_v2["acierto_claude"] = df_claude_k3_v2["acierto"]
df_resultados_k3_v2["acierto_gemini"] = df_gemini_k3_v2["acierto"]

df_resultados_k3_v2.to_csv("/content/drive/MyDrive/TFM/experimento_A_k3_v2_evaluado.csv", index=False)
print(df_resultados_k3_v2.shape)

(200, 14)


In [ ]:
df_resultados_a_k3_v2 = pd.read_csv("/content/drive/MyDrive/TFM/experimento_A_k3_v2_evaluado.csv")

In [ ]:
resumen_por_categoria = df_resultados_a_k3_v2.groupby("tamano_correo")["acierto_claude"].agg(["sum", "count", "mean"])
resumen_por_categoria.columns = ["aciertos", "total", "tasa_acierto"]
print("Claude:")
print(resumen_por_categoria)
print()

resumen_por_categoria = df_resultados_a_k3_v2.groupby("tamano_correo")["acierto_gemini"].agg(["sum", "count", "mean"])
resumen_por_categoria.columns = ["aciertos", "total", "tasa_acierto"]
print("Gemini:")
print(resumen_por_categoria)

Claude:
               aciertos  total  tasa_acierto
tamano_correo                               
corto                33     50      0.660000
intermedio           39     50      0.780000
largo                25     50      0.500000
multi_mensaje        22     30      0.733333
sin_respuesta        20     20      1.000000

Gemini:
               aciertos  total  tasa_acierto
tamano_correo                               
corto                35     50      0.700000
intermedio           37     50      0.740000
largo                22     50      0.440000
multi_mensaje        22     30      0.733333
sin_respuesta        20     20      1.000000


In [ ]:
largas_k3 = df_resultados_a_k3_v2.loc[100:149].copy()
largas_k3["necesito_chunking"] = largas_k3["id_correo"].apply(lambda x: n_tokens.loc[int(float(x))] > 256)

resumen_chunking_k3_claude = largas_k3.groupby("necesito_chunking")["acierto_claude"].agg(["sum", "count", "mean"])
resumen_chunking_k3_claude.columns = ["aciertos", "total", "tasa_acierto"]
print("Claude:")
print(resumen_chunking_k3_claude)
print()

resumen_chunking_k3_gemini = largas_k3.groupby("necesito_chunking")["acierto_gemini"].agg(["sum", "count", "mean"])
resumen_chunking_k3_gemini.columns = ["aciertos", "total", "tasa_acierto"]
print("Gemini:")
print(resumen_chunking_k3_gemini)

Claude:
                   aciertos  total  tasa_acierto
necesito_chunking                               
False                    12     15      0.800000
True                     13     35      0.371429

Gemini:
                   aciertos  total  tasa_acierto
necesito_chunking                               
False                    11     15      0.733333
True                     11     35      0.314286


K=5

In [ ]:
df_claude_k5_v2 = pd.read_csv("experimento_A_k5_v2_evaluado_claude.csv")
df_gemini_k5_v2 = pd.read_csv("experimento_A_k5_v2_evaluado_Gemini.csv")

print("Tasa de acierto Claude:", df_claude_k5_v2["acierto"].mean())
print("Tasa de acierto Gemini:", df_gemini_k5_v2["acierto"].mean())

coincidencias = (df_claude_k5_v2["acierto"] == df_gemini_k5_v2["acierto"]).sum()
print(f"Coinciden en {coincidencias}/200 filas")

discrepancias = df_claude_k5_v2[df_claude_k5_v2["acierto"] != df_gemini_k5_v2["acierto"]]
print(f"Discrepancias: {len(discrepancias)}")
print(discrepancias["id_pregunta"].tolist())

Tasa de acierto Claude: 0.72
Tasa de acierto Gemini: 0.725
Coinciden en 195/200 filas
Discrepancias: 5
[30, 45, 120, 137, 142]


In [ ]:
df_resultados_k5_v2 = df_claude_k5_v2.drop(columns=["acierto"]).copy()
df_resultados_k5_v2["acierto_claude"] = df_claude_k5_v2["acierto"]
df_resultados_k5_v2["acierto_gemini"] = df_gemini_k5_v2["acierto"]

df_resultados_k5_v2.to_csv("/content/drive/MyDrive/TFM/experimento_A_k5_v2_evaluado.csv", index=False)
print(df_resultados_k5_v2.shape)

(200, 14)


In [ ]:
df_resultados_a_k5_v2 = pd.read_csv("/content/drive/MyDrive/TFM/experimento_A_k5_v2_evaluado.csv")

In [ ]:
resumen_por_categoria = df_resultados_a_k5_v2.groupby("tamano_correo")["acierto_claude"].agg(["sum", "count", "mean"])
resumen_por_categoria.columns = ["aciertos", "total", "tasa_acierto"]
print("Claude:")
print(resumen_por_categoria)
print()

resumen_por_categoria = df_resultados_a_k5_v2.groupby("tamano_correo")["acierto_gemini"].agg(["sum", "count", "mean"])
resumen_por_categoria.columns = ["aciertos", "total", "tasa_acierto"]
print("Gemini:")
print(resumen_por_categoria)

Claude:
               aciertos  total  tasa_acierto
tamano_correo                               
corto                34     50      0.680000
intermedio           41     50      0.820000
largo                27     50      0.540000
multi_mensaje        22     30      0.733333
sin_respuesta        20     20      1.000000

Gemini:
               aciertos  total  tasa_acierto
tamano_correo                               
corto                36     50      0.720000
intermedio           41     50      0.820000
largo                26     50      0.520000
multi_mensaje        22     30      0.733333
sin_respuesta        20     20      1.000000


In [ ]:
largas_k5 = df_resultados_a_k5_v2.loc[100:149].copy()
largas_k5["necesito_chunking"] = largas_k5["id_correo"].apply(lambda x: n_tokens.loc[int(float(x))] > 256)

resumen_chunking_k5_claude = largas_k5.groupby("necesito_chunking")["acierto_claude"].agg(["sum", "count", "mean"])
resumen_chunking_k5_claude.columns = ["aciertos", "total", "tasa_acierto"]
print("Claude:")
print(resumen_chunking_k5_claude)
print()

resumen_chunking_k5_gemini = largas_k5.groupby("necesito_chunking")["acierto_gemini"].agg(["sum", "count", "mean"])
resumen_chunking_k5_gemini.columns = ["aciertos", "total", "tasa_acierto"]
print("Gemini:")
print(resumen_chunking_k5_gemini)

Claude:
                   aciertos  total  tasa_acierto
necesito_chunking                               
False                    12     15      0.800000
True                     15     35      0.428571

Gemini:
                   aciertos  total  tasa_acierto
necesito_chunking                               
False                    12     15           0.8
True                     14     35           0.4
